# Phase 4 — vLLM vs Hugging Face baseline

Same workloads as Phase 1 (single-request context sweep) and Phase 2 (batched sweep), run on **vLLM** with PagedAttention. The headline question: how much does an optimized serving system improve TTFT, throughput, and the (context × batch) feasibility frontier?

**Scope:**
- Model: `meta-llama/Meta-Llama-3.1-8B-Instruct` only — isolates the backend variable. (Phase 1 already varies the model; this notebook varies the backend.)
- Hardware: A100.
- Sweeps: ctx 8k/16k/32k/64k at batch 1 (mirrors Phase 1) plus ctx 8k & 32k at batch 1/2/4/8/16 (mirrors Phase 2).
- TensorRT-LLM is deferred — see CLAUDE.md.

**Results land in:** `results/phase4_llama31_a100.jsonl` (single file, both sweeps).

**Two semantic differences from HF that affect interpretation:**
1. **Peak GPU memory is ~constant across cells** — vLLM pre-allocates ~90% of GPU memory upfront for the KV cache pool. The HF-vs-vLLM peak-memory comparison is misleading; we drop it from the headline plots.
2. **vLLM doesn't OOM where HF did** — when the KV pool fills, vLLM serializes requests instead of failing. Cells that crashed in Phase 2 will succeed here, but with much higher total latency. That "soft degradation" is itself the finding.


In [ ]:
# 1. Setup. vLLM install is heavy (~3 min on Colab A100).
import os
os.chdir('/content/LLM_Inference')
!pip install -q -r requirements.txt
!pip install -q -r requirements-vllm.txt
print('cwd:', os.getcwd())


In [ ]:
# 2. Hugging Face login (Llama-3.1 is gated).
from huggingface_hub import login
login()


In [ ]:
# 3. Smoke test — verifies vLLM loads, RequestOutput.metrics is populated,
#    and TTFT comes back non-null. STOP if anything below errors or returns null.
!python3 scripts/run_vllm_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/llama3_1_8b_instruct.yaml \
  --context-lengths 1024 \
  --max-new-tokens 32 \
  --max-model-len 2048 \
  --results-path results/phase4_smoke.jsonl

import json
row = json.loads(open('results/phase4_smoke.jsonl').readlines()[-1])
assert row['success'], f"Smoke test failed: {row['error']}"
assert row['ttft_seconds'] is not None, "vLLM RequestOutput.metrics did not populate TTFT — check vLLM version / disable_log_stats"
print(f"smoke ok — ttft={row['ttft_seconds']:.3f}s total={row['total_latency_seconds']:.3f}s tps={row['tokens_per_second']:.2f}")


## Sweep A — single-request context length (mirrors Phase 1)

Same dimensions as `phase1_llama31_a100.jsonl` so the two can be plotted side by side.


In [ ]:
# 4. Context sweep on vLLM, batch=1, ctx 8k/16k/32k/64k.
#    max_model_len = 65k + decode budget so the 64k cell fits.
!python3 scripts/run_vllm_context_sweep.py --config configs/baseline_hf.yaml \
  --model-config configs/llama3_1_8b_instruct.yaml \
  --context-lengths 8192 16384 32768 65536 \
  --max-new-tokens 128 \
  --max-model-len 65664 \
  --results-path results/phase4_llama31_a100.jsonl


## Sweep B — batched inference (mirrors Phase 2)

Same `(context, batch)` grid as Phase 2 — the cells that OOM'd on HF are exactly the ones where vLLM should shine (or, at minimum, soft-degrade instead of failing).


In [ ]:
# 5. Batch sweep on vLLM. Appends to the same JSONL as the context sweep.
!python3 scripts/run_vllm_batch_experiment.py --config configs/baseline_hf.yaml \
  --model-config configs/llama3_1_8b_instruct.yaml \
  --context-lengths 8192 32768 \
  --batch-sizes 1 2 4 8 16 \
  --max-new-tokens 64 \
  --max-model-len 33024 \
  --results-path results/phase4_llama31_a100.jsonl


## Comparison plots

Loads:
- `results/phase1_llama31_a100.jsonl` — HF baseline, single-request context sweep
- `results/phase2_llama31_a100.jsonl` — HF baseline, batched sweep
- `results/phase4_llama31_a100.jsonl` — vLLM, both sweeps

Three figures:
1. **TTFT vs context length** (single-request). Direct apples-to-apples.
2. **Latency-vs-throughput Pareto** with both backends overlaid. **The headline plot.**
3. **Frontier annotation** — text+plot combination showing where HF OOM'd vs where vLLM degraded.

Peak GPU memory is intentionally not plotted (vLLM pre-allocates a pool, so the comparison would mislead).


In [ ]:
# 6. Load all three result files into one DataFrame, tagged by backend.
import json, glob
from pathlib import Path
import pandas as pd

def load_files(pattern, backend_label):
    rows = []
    for path in sorted(glob.glob(pattern)):
        for line in open(path):
            r = json.loads(line)
            r['_source'] = Path(path).name
            r['_backend'] = backend_label
            rows.append(r)
    return rows

hf_p1 = load_files('results/phase1_llama31_*.jsonl', 'HF (baseline)')
hf_p2 = load_files('results/phase2_llama31_*.jsonl', 'HF (baseline)')
vllm  = load_files('results/phase4_llama31_*.jsonl', 'vLLM')

df = pd.DataFrame(hf_p1 + hf_p2 + vllm)
df['per_request_tps'] = df.apply(
    lambda r: (r['tokens_per_second'] / r['batch_size']) if r['success'] else None, axis=1
)
print(f'loaded {len(df)} rows: HF={len(hf_p1)+len(hf_p2)}, vLLM={len(vllm)}')
print(df[['_backend', 'context_length', 'batch_size', 'success',
          'ttft_seconds', 'total_latency_seconds', 'tokens_per_second']]
      .sort_values(['_backend', 'context_length', 'batch_size'])
      .to_string(index=False))


In [ ]:
# 7. TTFT vs context length, single-request (batch=1) — both backends.
import matplotlib.pyplot as plt
from pathlib import Path

PLOT_DIR = Path('results/plots'); PLOT_DIR.mkdir(parents=True, exist_ok=True)

single = df[df.batch_size == 1].sort_values(['_backend', 'context_length'])
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for col, ylabel, ax in [
    ('ttft_seconds',          'TTFT (s)',                axes[0]),
    ('tokens_per_second',     'Throughput (tok/s)',      axes[1]),
    ('total_latency_seconds', 'Total latency (s)',       axes[2]),
]:
    for backend, grp in single.groupby('_backend'):
        ok = grp[grp.success == True].sort_values('context_length')
        ax.plot(ok.context_length, ok[col], marker='o', label=backend)
        fail = grp[grp.success == False]
        if len(fail):
            ax.scatter(fail.context_length, [0]*len(fail), marker='x', color='red', s=120, zorder=5, linewidths=2.5)
    ax.set_xscale('log', base=2)
    ax.set_xlabel('context length (tokens)')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle('Phase 4 — single-request: vLLM vs HF baseline (Llama-3.1-8B, A100)', fontsize=12, fontweight='bold')
plt.tight_layout()
out = PLOT_DIR / 'phase4_single_request.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
print(f'saved {out}')
plt.show()


In [ ]:
# 8. Latency-vs-throughput Pareto — the headline figure.
#    Each point is one (context, batch) cell. Up-and-right is strictly better.
import matplotlib.pyplot as plt

ok = df[df.success == True].copy()
fig, ax = plt.subplots(figsize=(9, 6.5))

# Use marker shape for context length, color for backend.
markers = {8192: 'o', 16384: 's', 32768: 'D', 65536: '^'}
colors  = {'HF (baseline)': '#1f77b4', 'vLLM': '#ff7f0e'}

for backend, grp_b in ok.groupby('_backend'):
    for ctx, grp_c in grp_b.groupby('context_length'):
        grp_c = grp_c.sort_values('batch_size')
        ax.plot(
            grp_c.tokens_per_second, grp_c.total_latency_seconds,
            marker=markers.get(ctx, 'o'), color=colors[backend],
            linestyle='-' if backend == 'vLLM' else '--',
            label=f'{backend}, ctx {ctx}',
        )
        for _, r in grp_c.iterrows():
            ax.annotate(f"B={int(r.batch_size)}",
                        (r.tokens_per_second, r.total_latency_seconds),
                        textcoords='offset points', xytext=(5, 4), fontsize=8)

ax.set_xlabel('Aggregate throughput (tokens/sec)  →  better')
ax.set_ylabel('Per-request latency (s)  →  worse')
ax.set_title('Phase 4 — Pareto: vLLM vs HF baseline')
ax.grid(alpha=0.3)
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
out = Path('results/plots/phase4_pareto.png')
plt.savefig(out, dpi=120, bbox_inches='tight')
print(f'saved {out}')
plt.show()


In [ ]:
# 9. Frontier annotation: which (context, batch) cells succeeded on each backend?
#    Side-by-side matrices so the reader can compare HF's hard-OOM frontier
#    against vLLM's mostly-green grid.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Restrict to the batch-sweep dimensions (ctx in {8192, 32768}, batch in 1..16).
batch_sweep = df[df.context_length.isin([8192, 32768]) & df.batch_size.isin([1, 2, 4, 8, 16])]

backends = sorted(batch_sweep['_backend'].unique())
fig, axes = plt.subplots(1, len(backends), figsize=(5.5 * len(backends), 3.5))
if len(backends) == 1:
    axes = [axes]

for ax, backend in zip(axes, backends):
    sub = batch_sweep[batch_sweep['_backend'] == backend]
    pivot = sub.pivot_table(index='context_length', columns='batch_size',
                            values='success', aggfunc='first')
    for i, ctx in enumerate(pivot.index):
        for j, bsz in enumerate(pivot.columns):
            v = pivot.loc[ctx, bsz]
            color = '#bde0a8' if v == True else ('#f4a8a8' if v == False else '#dddddd')
            ax.add_patch(Rectangle((j, i), 1, 1, facecolor=color, edgecolor='white'))
            label = 'OK' if v == True else ('OOM' if v == False else '-')
            ax.text(j + 0.5, i + 0.5, label, ha='center', va='center', fontsize=10)
    ax.set_xlim(0, len(pivot.columns)); ax.set_ylim(0, len(pivot.index))
    ax.invert_yaxis()
    ax.set_xticks(np.arange(len(pivot.columns)) + 0.5); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(np.arange(len(pivot.index)) + 0.5);   ax.set_yticklabels(pivot.index)
    ax.set_xlabel('batch size'); ax.set_ylabel('context length (tokens)')
    ax.set_title(backend)
    ax.tick_params(length=0)
    for spine in ax.spines.values(): spine.set_visible(False)

fig.suptitle('Phase 4 — feasibility frontier: HF (hard OOM) vs vLLM (PagedAttention)', fontsize=12, fontweight='bold')
plt.tight_layout()
out = Path('results/plots/phase4_frontier.png')
plt.savefig(out, dpi=120, bbox_inches='tight')
print(f'saved {out}')
plt.show()

# Note: where vLLM cells say 'OK' but total_latency is much larger than HF's
# single-batch baseline, the requests serialized — see latency in cell 8's
# Pareto plot. That's the "soft degradation" finding.
